<a href="https://colab.research.google.com/github/apmontesp/Landslides_-Applied-ML-Course/blob/main/notebooks/03_baseline_rf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 03: Baseline - Random Forest (Vía Google Drive)
**Proyecto de Maestría:** Detección de Deslizamientos | Universidad EAFIT

Este notebook establece el rendimiento base utilizando descriptores de textura HOG y Random Forest, leyendo directamente desde la unidad de Drive ya configurada.

## 1. Configuración de Acceso a Datos
Conectamos con Drive para evitar el uso de la API de Kaggle.

In [ ]:
from google.colab import drive
import os, h5py
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.auto import tqdm
from skimage.feature import hog
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report

# 1. Montar Drive
drive.mount('/content/drive')

# 2. Localizar datos (usando la ruta que ya validamos en el Notebook 02)
base_path = Path('/content/drive/MyDrive/Landslide4Sense')
detectar_img = list(base_path.glob('**/TrainData/img'))

if detectar_img:
    train_img_dir = detectar_img[0]
    train_mask_dir = detectar_img[0].parent / "mask"
    img_list = sorted(list(train_img_dir.glob("*.h5")))
    mask_list = sorted(list(train_mask_dir.glob("*.h5")))
    print(f"✅ Datos listos: {len(img_list)} muestras detectadas.")
else:
    print("❌ ERROR: No se encontró la carpeta en Drive.")

## 2. Extracción de Características (HOG)
HOG (Histogram of Oriented Gradients) captura las texturas de los deslizamientos (bordes, cicatrices en el terreno).

In [ ]:
N_SAMPLES = 1000 # Podemos empezar con 1000 para no agotar la RAM
X, y = [], []

for i in tqdm(range(min(N_SAMPLES, len(img_list))), desc="Extrayendo HOG"):
    with h5py.File(img_list[i], "r") as hf:
        patch = hf[list(hf.keys())[0]][()]
    with h5py.File(mask_list[i], "r") as hf:
        mask = hf[list(hf.keys())[0]][()]
    
    # Usamos bandas 4,3,2 (RGB) para el descriptor de textura
    rgb = patch[:,:,[3,2,1]]
    # Normalización rápida para HOG
    rgb = ((rgb - rgb.min()) / (rgb.ptp() + 1e-8) * 255).astype("uint8")
    
    # HOG: Orientación de gradientes en celdas de 16x16
    feats = hog(rgb, pixels_per_cell=(16,16), cells_per_block=(2,2), 
                channel_axis=-1, feature_vector=True)
    
    X.append(feats)
    y.append(int(mask.max() > 0)) # Clasificación: ¿Hay deslizamiento en este parche?

X, y = np.array(X), np.array(y)
print(f"✓ Dataset preparado: {X.shape}")

## 3. Entrenamiento y Evaluación
Utilizamos Random Forest para obtener nuestra métrica base.

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = []

for fold, (t_idx, v_idx) in enumerate(skf.split(X, y)):
    model = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)
    model.fit(X[t_idx], y[t_idx])
    
    preds = model.predict(X[v_idx])
    f1 = f1_score(y[v_idx], preds)
    scores.append(f1)
    print(f"Fold {fold+1} F1-Score: {f1:.4f}")

print(f"\n🚀 BASELINE FINAL: {np.mean(scores):.4f} ± {np.std(scores):.4f}")